# MODFLOW 6 API F: denitrification

This is notebook F of the API series. Notebook A ([`mf6-api-a-basic`](mf6-api-a-basic.ipynb)) covers the steps of an API run, and notebook C ([`mf6-api-c-recharge-callback`](mf6-api-c-recharge-callback.ipynb)) covers callbacks.

Use the API to add chemistry that MODFLOW 6 does not solve on its own. One simulation holds a groundwater flow (GWF) model and three groundwater transport (GWT) models, one for each form of nitrogen, which share one transport solution. Denitrification turns nitrate into nitrite and nitrite into dinitrogen gas,

$$\mathrm{NO_3} \xrightarrow{\;k_1\;} \mathrm{NO_2} \xrightarrow{\;k_2\;} \mathrm{N_2},$$

where $k_1$ and $k_2$ are first-order rate constants (1/T). Every time step, a callback reads the concentration of each species from MODFLOW's memory and writes the mass each reaction moves from one species to the next back to the models. A lake is the source of nitrate; nitrite and dinitrogen start at zero everywhere, so all of them comes from the reactions.

The flow and transport model is the MODFLOW 6 example `ex-gwt-prudic2004t2` (Prudic and others, 2004), in which streams (SFR), a lake (LAK), and the Water Mover (MVR) carry solute as well as water, through the Streamflow Transport (SFT), Lake Transport (LKT), and Mover Transport (MVT) packages. It works in feet and days, and concentrations are in milligrams per liter (mg/L).

> **Model reference.** Prudic, D.E., Konikow, L.F., and Banta, E.R., 2004, A new streamflow-routing (SFR1) package to simulate stream-aquifer interaction with MODFLOW-2000: U.S. Geological Survey Open-File Report 2004–1042, 95 p., https://doi.org/10.3133/ofr20041042.

## What this notebook covers

By the end of this notebook you will be able to:

- run several transport models in one simulation and move mass between them in a callback,
- write reaction rates to a Mass Source Loading (SRC) package, so they appear in every mass budget, and
- follow a chain of reactions in maps, concentrations through time, and the mass held in the aquifer.

## Imports

Import FloPy, NumPy, and Matplotlib, and `Callbacks` and `run_simulation` from `modflowapi`. `mf6_api_helpers` finds the MODFLOW 6 library, and `mf6_api_f` builds the flow and transport model, whose data are in `examples/data/ex-gwt-prudic2004t2`.

In [ ]:
%matplotlib inline

import math
import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import mf6_api_f as pm
import numpy as np
from flopy.plot.styles import styles
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.ticker import FuncFormatter, StrMethodFormatter
from mf6_api_helpers import find_mf6_libraries
from modflowapi import Callbacks, run_simulation

## Find the MODFLOW 6 library

The API runs MODFLOW 6 through its shared library, `libmf6` (`libmf6.so` on Linux, `libmf6.dylib` on macOS, and `libmf6.dll` on Windows), rather than through the `mf6` program. pixi installs both with the class MODFLOW 6. `find_mf6_libraries()` returns the path of each.

In [ ]:
lib_path, mf6_exe = find_mf6_libraries()
print(f"library: {lib_path.name}")
print(f"program: {mf6_exe.name}")

## The problem

Set the workspace and the transport time, and choose how the callback applies the reactions with `reaction_method`. The flow model is steady state. Transport runs for about 25 years in `nstp_transport` time steps, and the callback acts once each time step.

- `"src_rhs"`: the callback works out the mass each reaction moves and writes it to an SRC package for each species. MODFLOW adds the SRC terms to the transport equations and to the mass budget of each model, so every budget balances.
- `"operator_split"`: the callback changes the concentrations themselves at the end of each time step. It is simpler, but no package accounts for the mass the reactions move, so it shows up as an error in the mass budget of each model.

In [ ]:
sim_name = "nitrate-nitrite"

workspace = pl.Path("models/mf6-api-f-nitrate")

# reaction-coupling method: "src_rhs" (clean budget) or "operator_split"
reaction_method = "src_rhs"

# transport timing: flow is steady; transport runs ~25 years in nstp_transport
# equal steps, and the API callback fires once per step
total_time = 9131.0  # days (~25 years)
nstp_transport = 300

### Reaction settings

Each reaction is first order, set by a half-life. The mass of nitrite made from a unit mass of nitrate, `y1`, and the mass of dinitrogen made from a unit mass of nitrite, `y2`, come from the molar masses of the species.

The nitrate concentration of the lake is held at 50 mg/L for 10 years, lowered to 40 mg/L, just below the drinking-water limit, over the next 5 years, and held there to the end. Only the larger of the two lakes supplies nitrate.

In [ ]:
SUSTAINED_SOURCE = True  # True: lake held at a (time-varying) NO3; False: slug
no3_source_conc = 50.0  # initial lake nitrate concentration (mg/L)

# source-reduction time series: constant until year 10, reduced to just below the
# nitrate MCL by year 15, then held there. Concentrations here are as NO3, so the
# MCL is 44.3 mg/L - the 10 mg/L the regulation states is as N.
no3_source_conc_final = 40.0
source_reduce_start = 10.0 * 365.0  # days (10 years)
source_reduce_end = 15.0 * 365.0  # days (15 years)
source_ts = [
    (0.0, no3_source_conc),
    (source_reduce_start, no3_source_conc),
    (source_reduce_end, no3_source_conc_final),
    (total_time, no3_source_conc_final),
]

half_life_no3 = 730.0  # NO3 first-order half-life (days)
half_life_no2 = 365.0  # NO2 first-order half-life (days)
k1 = math.log(2.0) / half_life_no3  # NO3 -> NO2 rate (1/day)
k2 = math.log(2.0) / half_life_no2  # NO2 -> N2  rate (1/day)

# species molar masses (g/mol) and the mass yields from molar stoichiometry
M_NO3 = 62.004
M_NO2 = 46.005
M_N2 = 28.014
y1 = M_NO2 / M_NO3  # 1 mol NO3 -> 1 mol NO2  (~0.742 mass)
y2 = 0.5 * M_N2 / M_NO2  # 1 mol NO2 -> 0.5 mol N2 (~0.305 mass)

SPECIES = ["no3", "no2", "n2"]

# map display: mask concentrations below MASK_BELOW (log color scale) and draw the
# drinking-water maximum contaminant level (MCL) contour where one exists. EPA MCLs
# are 10 mg/L as N for nitrate (= 44.3 mg/L as NO3) and 1 mg/L as N for nitrite
# (= 3.28 mg/L as NO2); dinitrogen has none.
MASK_BELOW = 0.01
MCL = {"no3": 44.3, "no2": 3.28}

### Build the model

`mf6_api_f.build_prudic_simulation()` builds one simulation with the flow model and one transport model for each species in `SPECIES`, all sharing one transport solution and each joined to the flow model by a GWF-GWT exchange, so the callback reaches all three species in the same solution. With `reaction_method="src_rhs"`, each species also gets an SRC package with one entry for every active cell, which the callback fills in each time step.

In [ ]:
sim = pm.build_prudic_simulation(
    workspace,
    mf6_exe,
    SPECIES,
    reaction_method=reaction_method,
    source_species="no3",
    source_time_series=source_ts if SUSTAINED_SOURCE else None,
    slug_source_conc=no3_source_conc,
    total_time=total_time,
    nstp=nstp_transport,
)
sim.write_simulation(silent=True)
print(f"reaction method: {reaction_method}")
print("wrote simulation to", workspace)

## Apply the reactions with a callback

Two callbacks are defined below, and `reaction_method` chooses which one runs.

- **`SrcRhsCallback`** (`src_rhs`) acts at the start of each time step. It reads the concentrations in the aquifer and works out the mass of each reaction per day, $r_1 = k_1\,C_{\mathrm{NO_3}}\,V_p$ and $r_2 = k_2\,C_{\mathrm{NO_2}}\,V_p$, where $C$ is a concentration (M/L³) and $V_p$ is the saturated pore volume of the cell (L³). It writes them to the SRC loading of each species, the variable `SMASSRATE`: nitrate loses $r_1$, nitrite gains $y_1 r_1$ and loses $r_2$, and dinitrogen gains $y_2 r_2$.
- **`OperatorSplitCallback`** (`operator_split`) acts at the end of each time step and changes the concentrations directly: nitrate loses $1 - e^{-k_1 \Delta t}$ of its concentration, nitrite gains what that makes and loses its own share, and dinitrogen gains what nitrite makes.

The transport model's concentrations, `X`, hold the aquifer cells first and then the lakes and stream reaches, so the reactions are applied only to the first `naq` values. `mf6_api_f.saturated_pore_volume_reduced()` gives the pore volume of each cell.

In [ ]:
class OperatorSplitCallback:
    """Apply the NO3->NO2->N2 chain by overwriting the state arrays X at the end
    of each transport time step (operator splitting)."""

    def __init__(self):
        self.history = []
        self.elapsed = 0.0
        self._ptrs = None

    def __call__(self, sim, step):
        if step != Callbacks.timestep_end:
            return
        models = {m.name.lower(): m for m in sim.models}
        if not all(s in models for s in SPECIES):
            return
        if self._ptrs is None:
            mf6 = models["no3"].mf6
            self._ptrs = {
                s: mf6.get_value_ptr(mf6.get_var_address("X", models[s].name))
                for s in SPECIES
            }
            # X is [aquifer cells | LKT lakes | SFT reaches]; react the aquifer only
            self.naq = mf6.get_value(mf6.get_var_address("X", "FLOW")).shape[0]
        c, naq = self._ptrs, self.naq
        dt = sim.delt
        d1 = np.maximum(c["no3"][:naq], 0.0) * (1.0 - math.exp(-k1 * dt))  # NO3 lost
        d2 = np.maximum(c["no2"][:naq], 0.0) * (1.0 - math.exp(-k2 * dt))  # NO2 lost
        # in-place updates write straight into MF6 memory (do not rebind)
        c["no3"][:naq] -= d1
        c["no2"][:naq] += y1 * d1 - d2
        c["n2"][:naq] += y2 * d2
        self._record(dt, c, naq)

    def _record(self, dt, c, naq):
        self.elapsed += dt
        self.history.append((self.elapsed, *(float(c[s][:naq].sum()) for s in SPECIES)))


class SrcRhsCallback:
    """Compute the reaction mass rates from the current concentrations and write
    them into each species' SRC package at the start of each time step. The SRC
    terms enter the solution RHS and are booked in the GWT budget, so every
    per-model mass budget closes."""

    def __init__(self, cell_pore_volume):
        self.pv = cell_pore_volume  # porosity*volume per active node (reduced)
        self.history = []
        self.elapsed = 0.0
        self._x = None  # live concentration views
        self._smr = None  # live SRC SMASSRATE views
        self._red = None  # SRC entry index -> reduced node
        self._sat_addr = None

    def __call__(self, sim, step):
        if step != Callbacks.timestep_start:
            return
        models = {m.name.lower(): m for m in sim.models}
        if not all(s in models for s in SPECIES):
            return
        mf6 = models["no3"].mf6
        if self._x is None:
            self._x = {
                s: mf6.get_value_ptr(mf6.get_var_address("X", models[s].name))
                for s in SPECIES
            }
            # SMASSRATE (not BOUND) is the loading array MF6 uses to formulate the
            # SRC contribution to the solution
            self._smr = {
                s: mf6.get_value_ptr(
                    mf6.get_var_address("SMASSRATE", models[s].name, "SRC-1")
                )
                for s in SPECIES
            }
            # SRC entries were created in reduced-node order, but map explicitly
            # via NODELIST to be safe
            self._red = {
                s: mf6.get_value(
                    mf6.get_var_address("NODELIST", models[s].name, "SRC-1")
                ).astype(int)
                - 1
                for s in SPECIES
            }
            self._sat_addr = mf6.get_var_address("SAT", "FLOW", "NPF")
            # aquifer node count (X also carries LKT/SFT unknowns past this)
            self.naq = self.pv.shape[0]
            assert self._x["no3"][: self.naq].shape == self.pv.shape

        naq = self.naq
        sat = mf6.get_value(self._sat_addr)
        satpv = self.pv * sat  # saturated pore volume
        c = {s: self._x[s][:naq] for s in SPECIES}  # aquifer concentrations
        # first-order reaction mass rates (mass / time) per active node
        r1 = k1 * np.maximum(c["no3"], 0.0) * satpv  # NO3 -> NO2
        r2 = k2 * np.maximum(c["no2"], 0.0) * satpv  # NO2 -> N2
        rate = {
            "no3": -r1,  # nitrate consumed
            "no2": y1 * r1 - r2,  # nitrite produced then consumed
            "n2": y2 * r2,  # dinitrogen produced
        }
        for s in SPECIES:
            self._smr[s][:] = rate[s][self._red[s]]
        self._record(sim.delt, c)

    def _record(self, dt, c):
        self.elapsed += dt
        self.history.append((self.elapsed, *(float(c[s].sum()) for s in SPECIES)))

### Run the model through the API

Create the callback `reaction_method` chooses and pass it to `run_simulation()`.

In [ ]:
if reaction_method == "src_rhs":
    callback = SrcRhsCallback(pm.saturated_pore_volume_reduced(sim))
else:
    callback = OperatorSplitCallback()

run_simulation(lib_path, workspace, callback, verbose=False)

### Check the mass budgets

Read the percent discrepancy of the last time step from the listing file of each species. With `src_rhs` the reactions are SRC terms, so each budget should balance to a small fraction of a percent. With `operator_split`, the discrepancy of each budget is about the mass the reactions moved into or out of that species.

In [ ]:
print(f"Per-model mass-balance discrepancy (last time step), method={reaction_method}:")
for s in SPECIES:
    pct = None
    for line in (workspace / f"{s}.lst").read_text().splitlines():
        if "PERCENT DISCREPANCY" in line:
            try:
                pct = float(line.split("=")[-1])
            except ValueError:
                pass
    print(f"  {s}: percent discrepancy = {pct}")

### Maps of each species

Map each species after 10 years, the end of the full-strength source, in four of the eight model layers. Concentrations below `MASK_BELOW` are left blank, and the color scale is logarithmic. The white contour is the drinking-water limit where a species has one. The star in panel A is the cell plotted through time below.

In [ ]:
decades = [0.01, 0.1, 1.0, 10.0, 100.0]
plain = FuncFormatter(lambda v, _: f"{v:g}")

lakibd = pm.load_grid_data()[2]
gwf = sim.get_model("flow")
labels = {
    "no3": "Nitrate (NO$_3$)",
    "no2": "Nitrite (NO$_2$)",
    "n2": "Dinitrogen (N$_2$)",
}
map_layers = [0, 2, 4, 7]

# observation cell (layer 1, row 18, col 18), 0-based
obs = (0, 17, 17)
xc = gwf.modelgrid.xcellcenters[obs[1], obs[2]]
yc = gwf.modelgrid.ycellcenters[obs[1], obs[2]]
lake_cmap = ListedColormap(["#6ca6cd"])
lake_overlay = np.ma.masked_where(lakibd == 0, np.ones((pm.nrow, pm.ncol)))

# maps are shown at 10 years (closest transport output time)
_mt = np.array(sim.get_model("no3").output.concentration().times)
map_totim = float(_mt[np.argmin(np.abs(_mt - 10.0 * 365.0))])
map_year = map_totim / 365.0

for s in SPECIES:
    gwt = sim.get_model(s)
    conc = gwt.output.concentration().get_data(totim=map_totim)
    lakec = gwt.lak.output.concentration().get_data(totim=map_totim).flatten()
    il, jl = np.where(lakibd > 0)
    for i, j in zip(il, jl):
        conc[0, i, j] = lakec[lakibd[i, j] - 1]
    cdisp = np.ma.masked_where((conc >= 1.0e29) | (conc < MASK_BELOW), conc)
    vmax = max(float(conc[conc < 1.0e29].max()), MASK_BELOW * 10.0)
    norm = LogNorm(vmin=MASK_BELOW, vmax=vmax)
    with styles.USGSMap():
        fig, axs = plt.subplots(2, 2, figsize=(7, 9), dpi=150, constrained_layout=True)
        for ip, ilay in enumerate(map_layers):
            ax = axs.flatten()[ip]
            pmv = flopy.plot.PlotMapView(
                model=gwf,
                ax=ax,
                layer=ilay,
            )
            pmv.plot_inactive(color_noflow="0.8")
            if ilay == 0:
                pmv.plot_array(lake_overlay, cmap=lake_cmap, vmin=0, vmax=1, alpha=0.6)
            ca = pmv.plot_array(cdisp, norm=norm)
            pmv.plot_bc("CHD-1", color="royalblue")
            if s in MCL:
                cs = pmv.contour_array(
                    conc,
                    masked_values=[1.0e30],
                    colors="white",
                    linewidths=1.5,
                    levels=[MCL[s]],
                )
                ax.clabel(cs, fmt=f"{MCL[s]:g}", fontsize=7, colors="white")
            if ip == 0:
                ax.plot(
                    xc,
                    yc,
                    marker="*",
                    ms=11,
                    mfc="magenta",
                    mec="k",
                    mew=0.6,
                    zorder=6,
                    clip_on=False,
                )
                ax.annotate(
                    "(1, 18, 18)",
                    xy=(xc, yc),
                    xytext=(xc - 3400.0, yc + 2200.0),
                    textcoords="data",
                    ha="right",
                    va="center",
                    fontsize=7,
                    color="black",
                    zorder=7,
                    arrowprops=dict(
                        arrowstyle="-", color="black", lw=0.6, shrinkA=0, shrinkB=4
                    ),
                )
            styles.heading(
                ax=ax, letter=chr(ord("A") + ip), heading=f"Model layer {ilay + 1}"
            )
            ax.set_aspect("equal")
            ax.set_xlabel("x (ft)")
            ax.set_ylabel("y (ft)")
            ticks = [t for t in decades if MASK_BELOW * 0.999 <= t <= vmax * 1.001]
            cb = fig.colorbar(ca, ax=ax, shrink=0.5, ticks=ticks)
            cb.ax.yaxis.set_major_formatter(plain)
            cb.ax.minorticks_off()
        mcl_note = f"; white: drinking-water limit, {MCL[s]:g} mg/L" if s in MCL else ""
        fig.suptitle(
            f"{labels[s]} after {map_year:.0f} years (blue: constant head{mcl_note})",
            fontsize=11,
            fontweight="bold",
        )

**What to look for.** Nitrate covers the largest area and reaches the highest concentration, 48.6 mg/L, above the drinking-water limit of 44.3 mg/L drawn in white, near the lake that supplies it. Nitrite reaches 10.0 mg/L and dinitrogen 8.7 mg/L. Both stay below nitrate, because each is made only where the species before it in the chain has reacted.

### Concentration and mass through time

Panel A is the concentration of each species at the cell marked by the star, on a log scale so the smaller nitrite and dinitrogen concentrations show. Panel B is the total mass of each species dissolved in the aquifer.

In [ ]:
colors = {"no3": "#2a78d6", "no2": "#eb6834", "n2": "#1baf7a"}
times, mass = pm.total_mass_in_storage(sim, SPECIES)

with styles.USGSPlot():
    fig, (axa, axb) = plt.subplots(
        2, 1, sharex=True, figsize=(7, 4.5), dpi=150, constrained_layout=True
    )
    ymax = 0.0
    for s in SPECIES:
        cobj = sim.get_model(s).output.concentration()
        t = np.array(cobj.times) / 365.0
        series = cobj.get_alldata()[:, obs[0], obs[1], obs[2]]
        axa.plot(t, series, color=colors[s], label=labels[s])
        ymax = max(ymax, float(np.max(series)))
    axa.set_yscale("log")
    axa.set_ylim(0.01, 1.5 * ymax)
    axa.yaxis.set_major_formatter(plain)
    axa.set_ylabel("Concentration (mg/L)")
    styles.heading(ax=axa, letter="A", heading="Concentration at cell (1, 18, 18)")
    for s in SPECIES:
        axb.plot(times, mass[s], color=colors[s], label=labels[s])
    fig.legend(
        *axb.get_legend_handles_labels(),
        loc="outside lower center",
        ncols=3,
        frameon=False,
    )
    axb.set_ylim(bottom=0.0)
    axb.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
    axb.set_ylabel("Total mass in storage (kg)")
    styles.heading(ax=axb, letter="B", heading="Mass in aquifer storage")
    axb.set_xlim(0.0, 25.0)
    axb.set_xlabel("Time (years)")
    fig.align_ylabels((axa, axb))

**What to look for.** At the cell marked by the star, nitrate peaks at 46.5 mg/L after 2.3 years and falls to 37.2 mg/L by the end, following the source as it is lowered. Nitrite peaks at 1.13 mg/L after 1.1 years and dinitrogen at 0.024 mg/L after 1.2 years.

The mass in the aquifer (B) keeps rising after the concentration at the cell has leveled off: nitrate mass peaks after 10.3 years, nitrite after 11.3, and dinitrogen after 17.1. A single monitoring well shows when the front passed it; only the mass balance shows how much is still in the aquifer, which at the end of the run holds 308,000 kg of nitrate.

## Recap

You added a chain of reactions that MODFLOW 6 does not have to three transport models in one simulation. The `src_rhs` callback wrote the mass each reaction moves to an SRC package of each species every time step, so every mass budget balances, while `operator_split` changes the concentrations directly and leaves the reactions out of the budgets. To see the difference, set `reaction_method` to `"operator_split"` and run the notebook again from the cell that sets it.